## Распознавание эмоций по тексту
Датасет был найден на сайте Kaggle. Ссылка на датасет - https://www.kaggle.com/datasets/shreejitcheela/text-emotion-recognition

In [17]:
import numpy as np
import pandas as pd

import re
from tqdm import tqdm

import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

In [ ]:
# для установки всех необходимых пакетов nltk

'''import nltk
nltk.download()'''

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

## Загрузка датасета

In [2]:
full_data_path = fr"DataBases\text_emotions\Text_Emotion.csv"

In [3]:
full_df = pd.read_csv(full_data_path)
full_df

,text,emotion
0,carefully word blog posts amount criticism hea...,☹️
1,cannot remember little mermaid feeling carefre...,🙂
2,not feeling super well turns cold knocked next...,🙂
3,feel honored part group amazing talents,🙂
4,think helping also began feel pretty lonely lo...,☹️
...,...,...
282817,feel honored motivated share world life changi...,🙂
282818,feel like gloaty really delighted,🙂
282819,feel little energetic one day next several day...,🙂
282820,feel work experience fell although fantastic o...,🙂


In [4]:
full_df[full_df.loc[:, ['emotion']] == '☹️'] = 0
full_df[full_df.loc[:, ['emotion']] == '🙂'] = 1
full_df

,text,emotion
0,carefully word blog posts amount criticism hea...,0
1,cannot remember little mermaid feeling carefre...,1
2,not feeling super well turns cold knocked next...,1
3,feel honored part group amazing talents,1
4,think helping also began feel pretty lonely lo...,0
...,...,...
282817,feel honored motivated share world life changi...,1
282818,feel like gloaty really delighted,1
282819,feel little energetic one day next several day...,1
282820,feel work experience fell although fantastic o...,1


In [5]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 282822 entries, 0 to 282821
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   text     282822 non-null  object
 1   emotion  282822 non-null  object
dtypes: object(2)
memory usage: 4.3+ MB


In [6]:
full_df.emotion = full_df.emotion.astype('Int8')
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 282822 entries, 0 to 282821
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   text     282822 non-null  object
 1   emotion  282822 non-null  Int8  
dtypes: Int8(1), object(1)
memory usage: 2.7+ MB


In [7]:
full_df.isna().sum()  # нет Nan значений, что было показано при выводе info

text       0
emotion    0
dtype: int64

In [8]:
# примеры текста отрицательных эмоций
for true_value in full_df[full_df.emotion == 0].sample(5)['text']:
    print(true_value)

feel tad regretful
feeling quite heartbroken
feeling restlessness discontent lately
already feel like damaged goods like sort monster person broken vase
feel sad view say not ro bunch words


In [9]:
# примеры текста положительных эмоций
for true_value in full_df[full_df.emotion == 1].sample(5)['text'].head(5):
    print(true_value)

feel sincere loving search truth condemnation hierarchical church gay community not affected much people
feel pretty el dueto dito entre eminem nicki minaj pagetitle adictivoz
feel bit hopeful
feeling quite adventurous tried drinks normally read pages pocketbooks
not always feel completely welcomed two groups belong


Предложения не похожи на написанные человеком. Они не являются полноценными. Слова идут обрывками и полный смысл предложения не понятен.

### Гипотеза: текст уже был предобработан

In [10]:
def remove_punctuation(text):
    return "".join([ch if ch not in string.punctuation else ' ' for ch in text])

def remove_numbers(text):
    return ''.join([i if not i.isdigit() else ' ' for i in text])

def remove_multiple_spaces(text):
  """удаление специальных символов"""
  return re.sub(r'\s+', ' ', text, flags=re.I)

In [11]:
english_stop_words = stopwords.words('english')
snowball = SnowballStemmer(language='english')

def tokenize_sentence(sentence: str, remove_stop_words: bool = True):
    prep_text = [remove_multiple_spaces(remove_numbers(remove_punctuation(sentence.lower())))]
    prep_text = ' '.join(prep_text)
    
    tokens = word_tokenize(prep_text, language = 'english')
    if remove_stop_words:
        tokens = [i for i in tokens if i not in english_stop_words]
    tokens = [snowball.stem(i) for i in tokens]
    return tokens

In [13]:
for true_value in full_df[full_df.emotion == 1].sample(10)['text']:
    print(len(true_value.split()), len(tokenize_sentence(true_value)))

9 9
4 4
24 23
10 10
5 5
6 6
11 11
4 4
22 21
17 17


до обработки

In [14]:
text_length = full_df.text.apply(len).to_list()  # длины строк с текстом
text_word_amout = []  # количество слов в тексте (важно, тк при обработке происходит еще и стемминг)
for i in full_df.text:
    text_word_amout += [len(i.split())]

обработка

In [15]:
english_stopwords = stopwords.words("english")

In [18]:
prep_texts = []
for t in tqdm(full_df.text):
    prep_texts.append(' '.join(tokenize_sentence(t)))

100%|██████████| 282822/282822 [00:29<00:00, 9740.08it/s] 


In [19]:
full_df['prep_text'] = prep_texts
full_df

,text,emotion,prep_text
0,carefully word blog posts amount criticism hea...,0,care word blog post amount critic hear place c...
1,cannot remember little mermaid feeling carefre...,1,rememb littl mermaid feel carefre beauti life ...
2,not feeling super well turns cold knocked next...,1,feel super well turn cold knock next three wee...
3,feel honored part group amazing talents,1,feel honor part group amaz talent
4,think helping also began feel pretty lonely lo...,0,think help also began feel pretti lone lot peo...
...,...,...,...
282817,feel honored motivated share world life changi...,1,feel honor motiv share world life chang gift a...
282818,feel like gloaty really delighted,1,feel like gloati realli delight
282819,feel little energetic one day next several day...,1,feel littl energet one day next sever day hard...
282820,feel work experience fell although fantastic o...,1,feel work experi fell although fantast opportu...


In [20]:
text_length2 = full_df.prep_text.apply(len).to_list()
text_word_amout2 = []  # количество слов в тексте (важно, тк при обработке происходит еще и стемминг)
for i in full_df.text:
    text_word_amout2 += [len(i.split())]

In [21]:
mean_len_diff = np.mean([abs(text_length2[i] - text_length[i]) for i in range(len(text_length))])
mean_amount_diff = np.mean([abs(text_word_amout2[i] - text_word_amout[i]) for i in range(len(text_word_amout))])

In [22]:
print(f'средняя разница длин строк до и после обработки: {mean_len_diff}')
print(f'средняя разница количества слов в строках до и после обработки: {mean_amount_diff}')

средняя разница длин строк до и после обработки: 8.757479969733613
средняя разница количества слов в строках до и после обработки: 0.0


Вывод: текст скорее всего уже был обработан по двум причинам: средняя разница количества слов в строках до и после обработки равна 0, предложения являются несвязными. Но обработка была другая, возможно, с использованием других библиотек. Обработка была неполной. К примеру, не был сделан стемминг и лемматизация

Основываясь на том, что гипотеза не была точно подтверждена или опровергнута, будут созданы несколько вариантов датасета: 
1) удаление специальных символов, чисел и пунктуации, стоп слов
2) удаление специальных символов, чисел и пунктуации, стоп слов + стемминг

В датасете не будет изначально проведена лемматизация, потому что это очень времязатратный процесс. Выдвигается новая гипотеза: отсутствие лемматизации не скажется на качестве модели.

## Создание датасета с различными степенями обработки данных

полностью обработанный датасет уже создан, просто переименовываем столбец с ним

In [35]:
full_df.rename(columns={'prep_text': 'full_prep_text'}, inplace=True)

In [36]:
def tokenize_sentence(sentence: str, remove_stop_words: bool = True):
    prep_text = [remove_multiple_spaces(remove_numbers(remove_punctuation(sentence.lower())))]
    prep_text = ' '.join(prep_text)
    
    tokens = word_tokenize(prep_text, language = 'english')
    if remove_stop_words:
        tokens = [i for i in tokens if i not in english_stop_words]
    return tokens

In [37]:
prep_text = [' '.join(tokenize_sentence(text)) for text in full_df.text]

In [38]:
full_df['no_stem_text'] = prep_text

In [39]:
full_df

,text,emotion,full_prep_text,no_stem_text
0,carefully word blog posts amount criticism hea...,0,care word blog post amount critic hear place c...,carefully word blog posts amount criticism hea...
1,cannot remember little mermaid feeling carefre...,1,rememb littl mermaid feel carefre beauti life ...,remember little mermaid feeling carefree beaut...
2,not feeling super well turns cold knocked next...,1,feel super well turn cold knock next three wee...,feeling super well turns cold knocked next thr...
3,feel honored part group amazing talents,1,feel honor part group amaz talent,feel honored part group amazing talents
4,think helping also began feel pretty lonely lo...,0,think help also began feel pretti lone lot peo...,think helping also began feel pretty lonely lo...
...,...,...,...,...
282817,feel honored motivated share world life changi...,1,feel honor motiv share world life chang gift a...,feel honored motivated share world life changi...
282818,feel like gloaty really delighted,1,feel like gloati realli delight,feel like gloaty really delighted
282819,feel little energetic one day next several day...,1,feel littl energet one day next sever day hard...,feel little energetic one day next several day...
282820,feel work experience fell although fantastic o...,1,feel work experi fell although fantast opportu...,feel work experience fell although fantastic o...


In [40]:
full_df.dropna(inplace=True)

## Промежуточное сохранение
чтобы каждый раз не делать обработку, а просто загружать готовый датасет

In [42]:
full_df.to_csv('DataBases/prepared_datasets/full_dataset_emotions_text.csv', index=False)

In [43]:
full_df = pd.read_csv('DataBases/prepared_datasets/full_dataset_emotions_text.csv')
full_df

,text,emotion,full_prep_text,no_stem_text
0,carefully word blog posts amount criticism hea...,0,care word blog post amount critic hear place c...,carefully word blog posts amount criticism hea...
1,cannot remember little mermaid feeling carefre...,1,rememb littl mermaid feel carefre beauti life ...,remember little mermaid feeling carefree beaut...
2,not feeling super well turns cold knocked next...,1,feel super well turn cold knock next three wee...,feeling super well turns cold knocked next thr...
3,feel honored part group amazing talents,1,feel honor part group amaz talent,feel honored part group amazing talents
4,think helping also began feel pretty lonely lo...,0,think help also began feel pretti lone lot peo...,think helping also began feel pretty lonely lo...
...,...,...,...,...
282817,feel honored motivated share world life changi...,1,feel honor motiv share world life chang gift a...,feel honored motivated share world life changi...
282818,feel like gloaty really delighted,1,feel like gloati realli delight,feel like gloaty really delighted
282819,feel little energetic one day next several day...,1,feel littl energet one day next sever day hard...,feel little energetic one day next several day...
282820,feel work experience fell although fantastic o...,1,feel work experi fell although fantast opportu...,feel work experience fell although fantastic o...
